# Auditoría Mecanística y Validación de Fidelidad XAI en Detección de Sensacionalismo
**Modelo:**  (BETO fine-tuned)  
**Hardware:** GPU NVIDIA GeForce GTX 1650 (CUDA)  
**Muestra Experimental:** 40 Noticias Balanceadas (20 IA Sintético, 20 Amarillismo Real; 10 Sensacionalistas y 10 No Sensacionalistas por fuente)  
**Batería de Pruebas:**
1. Fidelidad Causal: *Comprehensiveness* (Erasure) y *Sufficiency*
2. Perturbación Ablativa: Curvas MoRF (*Most Relevant First*) y LoRF (*Least Relevant First*)
3. Sanity Check de Parámetros: Aleatorización en cascada de capas (Adebayo et al., 2018)
4. **Análisis de Inversión de Decisión:** Comparativa de cambio de clase predicha bajo 3 estrategias de perturbación (*Enmascaramiento [MASK]*, *Eliminación de Tokens*, *Ruido Vocabulario Aleatorio*)
5. Mapas de calor de atribución a nivel de subword tokens


In [ ]:
import os
import gc
import time
import json
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients, InputXGradient
import shap
from lime.lime_text import LimeTextExplainer

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[INIT] Dispositivo configurado: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'[INIT] GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
MODEL_NAME = 'JJNeila/bert-spanish-sensationalism-oss'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, attn_implementation='eager')
model = model.to(DEVICE)
model.eval()
LABEL_NAMES = ['No Sensacionalista', 'Sensacionalista']
print('[MODEL] Modelo y tokenizador cargados exitosamente en VRAM.')


In [ ]:
PATH_AMA = '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/Dataset/dataset_Amarillismo.csv'
PATH_IA = '/home/ubuntu/Documentos/Tesis/Datasets/sensacionalismo/dataset_IA_sintetico_70.csv'

df_ama_full = pd.read_csv(PATH_AMA)
df_ia_full = pd.read_csv(PATH_IA)

# 1. Muestra IA Sintético (10 Sensacionalistas, 10 No Sensacionalistas)
ia_sens_idx = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
ia_nonsens_idx = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]

samples_ia = []
for idx in ia_sens_idx:
    samples_ia.append({
        'id': f'IA_SENS_{idx}',
        'fuente': 'IA Sintético (70 reg.)',
        'texto': df_ia_full.loc[idx, 'Texto'],
        'clase_real': 1,
        'etiqueta_real': 'Sensacionalista',
        'dominio': df_ia_full.loc[idx, 'Tema']
    })
for idx in ia_nonsens_idx:
    samples_ia.append({
        'id': f'IA_NONSENS_{idx}',
        'fuente': 'IA Sintético (70 reg.)',
        'texto': df_ia_full.loc[idx, 'Texto'],
        'clase_real': 0,
        'etiqueta_real': 'No Sensacionalista',
        'dominio': df_ia_full.loc[idx, 'Tema']
    })

# 2. Muestra Amarillismo Real (10 Sensacionalistas, 10 No Sensacionalistas)
ama_sens_rows = df_ama_full[df_ama_full['Amarillismo'] == 'Amarillista'].iloc[:10]
ama_nonsens_rows = df_ama_full[df_ama_full['Amarillismo'] == 'No Amarillista'].iloc[[0, 1, 4, 5, 6, 2, 3, 7, 8, 9]]

samples_ama = []
for idx, r in ama_sens_rows.iterrows():
    samples_ama.append({
        'id': f'AMA_SENS_{idx}',
        'fuente': 'Amarillismo Real (202 reg.)',
        'texto': r['Titular'],
        'clase_real': 1,
        'etiqueta_real': 'Sensacionalista',
        'dominio': str(r.get('Fuente', 'Prensa'))
    })
for idx, r in ama_nonsens_rows.iterrows():
    samples_ama.append({
        'id': f'AMA_NONSENS_{idx}',
        'fuente': 'Amarillismo Real (202 reg.)',
        'texto': r['Titular'],
        'clase_real': 0,
        'etiqueta_real': 'No Sensacionalista',
        'dominio': str(r.get('Fuente', 'Prensa'))
    })

df_eval = pd.DataFrame(samples_ia + samples_ama)
print(f'Total registros cargados: {len(df_eval)}')
print(df_eval.groupby(['fuente', 'etiqueta_real']).size())


In [ ]:
texts = df_eval['texto'].tolist()
inputs_batch = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    logits_base = model(**inputs_batch).logits
    probs_base = torch.softmax(logits_base, dim=-1).cpu().numpy()

df_eval['prob_no_sens'] = probs_base[:, 0]
df_eval['prob_sens'] = probs_base[:, 1]
df_eval['clase_pred'] = np.argmax(probs_base, axis=1)
df_eval['pred_correcta'] = df_eval['clase_real'] == df_eval['clase_pred']

print(f'Aciertos Globales: {df_eval["pred_correcta"].sum()}/40 ({df_eval["pred_correcta"].mean()*100:.1f}%)')
print(f'Aciertos IA Sintético: {df_eval[df_eval["fuente"].str.contains("IA")]["pred_correcta"].sum()}/20')
print(f'Aciertos Amarillismo Real: {df_eval[df_eval["fuente"].str.contains("Amarillismo")]["pred_correcta"].sum()}/20')


In [ ]:
def predict_proba_texts(text_list):
    if isinstance(text_list, np.ndarray): text_list = text_list.tolist()
    elif isinstance(text_list, str): text_list = [text_list]
    encoded = tokenizer(text_list, padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        return torch.softmax(model(**encoded).logits, dim=-1).cpu().numpy()

embedding_layer = model.bert.embeddings.word_embeddings
def forward_with_embeds(embeds, attention_mask):
    return model(inputs_embeds=embeds, attention_mask=attention_mask).logits

ig_engine = IntegratedGradients(forward_with_embeds)
ixg_engine = InputXGradient(forward_with_embeds)
lime_engine = LimeTextExplainer(class_names=LABEL_NAMES, random_state=42)
shap_masker = shap.maskers.Text(tokenizer)
shap_engine = shap.Explainer(predict_proba_texts, shap_masker, output_names=LABEL_NAMES)


In [ ]:
def compute_faithfulness(text, tokens, attr_vec, target_class, top_ratio=0.20):
    seq_len = len(tokens)
    mask_id = tokenizer.mask_token_id
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        p_orig = torch.softmax(model(**inputs).logits, dim=-1)[0, target_class].item()
    content_idx = [i for i in range(1, seq_len-1) if tokens[i] not in ['[CLS]', '[SEP]', '[PAD]']]
    if not content_idx: return 0.0, 0.0
    scores = [(i, attr_vec[i]) for i in content_idx]
    k = max(1, int(round(len(content_idx) * top_ratio)))
    top_k = set([idx for idx, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]])
    
    # Erasure (Comprehensiveness)
    ids_e = inputs['input_ids'].clone()
    for idx in top_k: ids_e[0, idx] = mask_id
    with torch.no_grad():
        p_e = torch.softmax(model(input_ids=ids_e, attention_mask=inputs['attention_mask']).logits, dim=-1)[0, target_class].item()
    comp = p_orig - p_e
    
    # Sufficiency
    ids_s = inputs['input_ids'].clone()
    for idx in content_idx:
        if idx not in top_k: ids_s[0, idx] = mask_id
    with torch.no_grad():
        p_s = torch.softmax(model(input_ids=ids_s, attention_mask=inputs['attention_mask']).logits, dim=-1)[0, target_class].item()
    suff = p_orig - p_s
    return comp, suff

def compute_perturbation_flips(text, tokens, attr_vec, target_class, top_ratio=0.20, seed=42):
    seq_len = len(tokens)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        logits_orig = model(**inputs).logits
        pred_orig = torch.argmax(logits_orig, dim=-1).item()
    content_idx = [i for i in range(1, seq_len-1) if tokens[i] not in ['[CLS]', '[SEP]', '[PAD]']]
    if not content_idx: return False, False, False
    scores = [(i, attr_vec[i]) for i in content_idx]
    k = max(1, int(round(len(content_idx) * top_ratio)))
    top_k = set([idx for idx, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]])
    
    # 1. Enmascaramiento
    ids_m = inputs['input_ids'].clone()
    for idx in top_k: ids_m[0, idx] = tokenizer.mask_token_id
    with torch.no_grad():
        pred_m = torch.argmax(model(input_ids=ids_m, attention_mask=inputs['attention_mask']).logits, dim=-1).item()
    
    # 2. Eliminación
    keep_idx = [i for i in range(seq_len) if i not in top_k]
    with torch.no_grad():
        pred_d = torch.argmax(model(input_ids=inputs['input_ids'][:, keep_idx], attention_mask=inputs['attention_mask'][:, keep_idx]).logits, dim=-1).item()
        
    # 3. Ruido Aleatorio
    rng = np.random.RandomState(seed)
    rnd_tokens = rng.randint(6, tokenizer.vocab_size, size=len(top_k))
    ids_n = inputs['input_ids'].clone()
    for idx, rtok in zip(sorted(list(top_k)), rnd_tokens): ids_n[0, idx] = int(rtok)
    with torch.no_grad():
        pred_n = torch.argmax(model(input_ids=ids_n, attention_mask=inputs['attention_mask']).logits, dim=-1).item()
        
    return (pred_m != pred_orig), (pred_d != pred_orig), (pred_n != pred_orig)


In [ ]:
from IPython.display import Image, display

img_paths = [
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_faithfulness_comprehensiveness_sufficiency.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_curvas_morf_lorf_comparativa.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_parameter_randomization_adebayo.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_latencia_por_metodo.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_cambio_clase_perturbaciones_ia.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_cambio_clase_perturbaciones_amarillismo.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_cambio_clase_perturbaciones_comparativa_global.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_heatmap_ia_sensacionalista.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_heatmap_ia_no_sensacionalista.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_heatmap_amarillismo_real_sensacionalista.png',
    '/home/ubuntu/Documentos/Tesis/Modelos_Individuales/Sensacionalismo/reportes/imagenes/XAI_pruebas_sensacionalismo/xai_heatmap_amarillismo_real_no_sensacionalista.png'
]

for p in img_paths:
    local_p = os.path.join('../reportes/imagenes/XAI_pruebas_sensacionalismo', os.path.basename(p))
    target_p = local_p if os.path.exists(local_p) else p
    if os.path.exists(target_p):
        print(f'Desplegando: {os.path.basename(target_p)}')
        display(Image(filename=target_p))
